# Active case finding in Vietnam: run the analysis

Calibrates the homogeneous-mixing model, runs every arm in `vacf.economics.study` at each of
`N_DRAWS` posterior draws, and stores the outputs under `outputs/<RUN_NAME>/`. It calls
`analysis.run_and_store`, the same function as `scripts/massive/calibrate.py`, so a local
run and a MASSIVE run have the same layout. Load and plot a run with
[11-ACF-CEA-Results](11-ACF-CEA-Results.ipynb).

Twelve parameters are estimated by NUTS: the upstream model's sixteen, less the four that
only shape a contact-data-derived mixing matrix, which are unidentifiable under homogeneous
age mixing. The Canberra mixing term is dropped from the likelihood for the same reason.

Production runs belong on MASSIVE. Locally, 100 warmup and 100 samples on 8 chains took
about 40 minutes and did not converge (worst r_hat 1.28): enough to develop the results
notebook, not to report.

Restart the kernel and run from the top: the number of devices is fixed when JAX is first
imported.

In [ ]:
# Must run before JAX is imported, or NumPyro runs the chains one after another.
import os

N_CHAINS = 8

flags = [f for f in os.environ.get("XLA_FLAGS", "").split()
         if not f.startswith("--xla_force_host_platform_device_count=")]
os.environ["XLA_FLAGS"] = " ".join([f"--xla_force_host_platform_device_count={N_CHAINS}", *flags])

import jax

if jax.local_device_count() < N_CHAINS:
    raise RuntimeError(
        f"JAX sees {jax.local_device_count()} device(s). Restart the kernel and run this "
        "cell first, otherwise the chains run sequentially."
    )
print(f"devices: {jax.local_device_count()}")

In [ ]:
import logging
import warnings
from datetime import UTC, datetime

from vacf.constants import OUTPUT_PATH
from vacf.economics import analysis, study

RUN_NAME = f"local-{datetime.now(UTC):%Y%m%dT%H%MZ}"
N_RUNS = 100  # warmup iterations, and retained samples, per chain
N_DRAWS = 100  # posterior draws run through every arm
SEED = 0

logger = logging.getLogger("acf")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

print(study.describe())
print(f"run {RUN_NAME}: {N_CHAINS} chains, {N_RUNS} warmup + {N_RUNS} samples, {N_DRAWS} draws")
banner = study.provisional_warning()
if banner:
    warnings.warn(banner, stacklevel=1)

In [ ]:
RUN = OUTPUT_PATH / RUN_NAME
outputs = analysis.run_and_store(
    RUN, n_runs=N_RUNS, n_chains=N_CHAINS, n_draws=N_DRAWS, seed=SEED, logger=logger
)

In [ ]:
summary = outputs["posterior_summary"]
worst_rhat = summary["r_hat"].max()
print(f"wrote {RUN}")
print(f"worst r_hat {worst_rhat:.3f}")
if worst_rhat > 1.01:
    print("NOT CONVERGED. Use this run for development only.")
summary[["mean", "sd", "ess_bulk", "r_hat"]]

To inspect this run, open [11-ACF-CEA-Results](11-ACF-CEA-Results.ipynb) and set
`RUN = OUTPUT_PATH / "<RUN_NAME>"`, using the name printed above.